<a href="https://colab.research.google.com/github/jeremydouti2-hub/Data-Science-Projects/blob/main/PRODUCT_REVIEW_INTELLIGENCE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

In [ ]:
!kaggle datasets download -d snap/amazon-fine-food-reviews

In [ ]:
!kaggle datasets download -d paramaggarwal/amazon-product-images-dataset

In [ ]:
import zipfile
import os

#  Check downloaded files
print("Available files:", os.listdir())

#  Extract all ZIP files automatically
for file in os.listdir():
    if file.endswith(".zip"):
        print(f"Extracting: {file}")
        with zipfile.ZipFile(file, 'r') as zip_ref:
            zip_ref.extractall(file.replace(".zip", ""))

In [ ]:
print("Folders after extraction:", os.listdir())

In [ ]:
import os

# Check reviews folder
if os.path.exists("reviews_data"):
    print("Reviews files:", os.listdir("reviews_data"))
else:
    print("reviews_data folder not found")

# Check images folder safely
if os.path.exists("images_data"):
    print("Images files:", os.listdir("images_data"))
else:
    print("images_data folder not found")

In [ ]:
import pandas as pd

# Load CSV file
df = pd.read_csv("reviews_data/Reviews.csv")

# Keep only useful columns
df = df[['Text', 'Score']]

# Rename columns for clarity
df.columns = ['review_text', 'rating']

# Display basic info
print(df.shape)
df.head()

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()  # lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text

# Apply cleaning
df['clean_text'] = df['review_text'].apply(clean_text)

df[['review_text', 'clean_text']].head()

In [ ]:
!pip install textblob

In [ ]:
from textblob import TextBlob

def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0.1:
        return "positive"
    elif polarity < -0.1:
        return "negative"
    else:
        return "neutral"

# Apply sentiment analysis
df['sentiment'] = df['clean_text'].apply(get_sentiment)

df[['clean_text', 'sentiment']].head()

In [ ]:
def rating_to_sentiment(score):
    if score >= 4:
        return "positive"
    elif score <= 2:
        return "negative"
    else:
        return "neutral"

df['true_sentiment'] = df['rating'].apply(rating_to_sentiment)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(df['true_sentiment'], df['sentiment'])
print("Sentiment Analysis Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(df['true_sentiment'], df['sentiment'])
print("Sentiment Analysis Accuracy:", accuracy)

In [ ]:
import matplotlib.pyplot as plt

df['sentiment'].value_counts().plot(kind='bar')
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

In [ ]:
# Define keyword dictionary
categories = {
    "food": ["taste", "flavor", "delicious", "food", "eat"],
    "service": ["service", "staff", "support", "help"],
    "quality": ["quality", "good", "bad", "excellent"],
}

def classify_text(text):
    scores = {cat: 0 for cat in categories}

    for cat, keywords in categories.items():
        for word in keywords:
            if word in text:
                scores[cat] += 1

    # Return category with highest score
    return max(scores, key=scores.get)

# Apply classification
df['keyword_category'] = df['clean_text'].apply(classify_text)

df[['clean_text', 'keyword_category']].head()

In [ ]:
import matplotlib.pyplot as plt

df['keyword_category'].value_counts().plot(kind='bar')
plt.title("Keyword-Based Category Distribution")
plt.xlabel("Category")
plt.ylabel("Count")
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Convert text to numerical features
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_text'])

# Use rating-based sentiment as target
y = df['true_sentiment']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = MultinomialNB()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Naive Bayes Accuracy:", accuracy)

In [ ]:
print("Keyword Categories:")
print(df['keyword_category'].value_counts())

print("\nML Model Accuracy:", accuracy)

In [ ]:
import os

# Explore extracted folders
for root, dirs, files in os.walk("."):
    if len(files) > 0:
        print(root)
        break

In [ ]:
image_folder = "images_data"  # change this if needed

In [ ]:
from sklearn.datasets import load_sample_images
from PIL import Image
import numpy as np

#  Load sample images (no download needed)
dataset = load_sample_images()
images_raw = dataset.images

print("Number of images loaded:", len(images_raw))

In [ ]:
image_size = (100, 100)

images = []

for img in images_raw:
    pil_img = Image.fromarray(img)
    pil_img = pil_img.resize(image_size)

    images.append(np.array(pil_img))

print("Processed images:", len(images))

In [ ]:
features = []

for img in images:
    avg_color = np.mean(img, axis=(0,1))
    brightness = np.mean(img)
    contrast = np.std(img)

    features.append([
        avg_color[0],
        avg_color[1],
        avg_color[2],
        brightness,
        contrast
    ])

features = np.array(features)
print("Feature shape:", features.shape)

In [ ]:
def classify_image(feature):
    r, g, b, brightness, contrast = feature

    if r > g and r > b:
        return "food"
    elif b > r and b > g:
        return "electronics"
    elif g > r and g > b:
        return "nature"
    else:
        return "other"

image_categories = [classify_image(f) for f in features]

print(image_categories)

In [ ]:
features = []

for img in images:
    avg_color = np.mean(img, axis=(0,1))   # RGB mean
    brightness = np.mean(img)
    contrast = np.std(img)

    features.append([
        avg_color[0],  # Red
        avg_color[1],  # Green
        avg_color[2],  # Blue
        brightness,
        contrast
    ])

features = np.array(features)

print("Feature shape:", features.shape)

In [ ]:
def classify_image(feature):
    r, g, b, brightness, contrast = feature

    if r > g and r > b:
        return "food"
    elif b > r and b > g:
        return "electronics"
    elif g > r and g > b:
        return "nature"
    else:
        return "other"

image_categories = [classify_image(f) for f in features]

print(image_categories[:10])

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df_images = pd.DataFrame(image_categories, columns=["category"])

df_images['category'].value_counts().plot(kind='bar')
plt.title("Image Category Distribution")
plt.xlabel("Category")
plt.ylabel("Count")
plt.show()

In [ ]:
# Convert sentiment to numeric
sentiment_map = {"positive": 1, "neutral": 0, "negative": -1}

df['sentiment_num'] = df['sentiment'].map(sentiment_map)

# Basic text features
df['text_length'] = df['clean_text'].apply(len)
df['word_count'] = df['clean_text'].apply(lambda x: len(x.split()))

text_features = df[['sentiment_num', 'text_length', 'word_count']].values

print("Text features shape:", text_features.shape)

In [ ]:
print("Image features shape:", features.shape)

In [ ]:
min_size = min(len(text_features), len(features))

X_text = text_features[:min_size]
X_image = features[:min_size]

# Combine both
import numpy as np
X_combined = np.hstack((X_text, X_image))

print("Combined features shape:", X_combined.shape)

In [ ]:
y = df['true_sentiment'][:min_size]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy_combined = accuracy_score(y_test, y_pred)

print("Multimodal Accuracy:", accuracy_combined)

In [ ]:
print("Text Model Accuracy (from Section 2):", accuracy)
print("Multimodal Model Accuracy:", accuracy_combined)

In [ ]:
import pandas as pd

combined_df = pd.DataFrame(X_combined, columns=[
    "sentiment_num", "text_length", "word_count",
    "avg_red", "avg_green", "avg_blue", "brightness", "contrast"
])

correlation_matrix = combined_df.corr()

print(correlation_matrix)